# Installing Required Packages


In [1]:
!pip3 install sqlalchemy

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
!pip3 install psycopg2

# Importing the Required Packages

In [2]:
#Importing Package
import sqlalchemy
#Database Utility Class
from sqlalchemy.engine import create_engine
# Provides executable SQL expression construct
from sqlalchemy.sql import text
sqlalchemy.__version__

'2.0.36'

In [3]:
class PostgresqlDB:
    def __init__(self,user_name,password,host,port,db_name):
        """
        class to implement DDL, DQL and DML commands,
        user_name:- username
        password:- password of the user
        host
        port:- port number
        db_name:- database name
        """
        self.user_name = user_name
        self.password = password
        self.host = host
        self.port = port
        self.db_name = db_name
        self.engine = self.create_db_engine()

    def create_db_engine(self):
        """
        Method to establish a connection to the database, will return an instance of Engine
        which can used to communicate with the database
        """
        try:
            db_uri = f"postgresql+psycopg2://{self.user_name}:{self.password}@{self.host}:{self.port}/{self.db_name}"
            return create_engine(db_uri)
        except Exception as err:
            raise RuntimeError(f'Failed to establish connection -- {err}') from err

    def execute_dql_commands(self,stmnt,values=None):
        """
        DQL - Data Query Language
        SQLAlchemy execute query by default as

        BEGIN
        ....
        ROLLBACK

        BEGIN will be added implicitly everytime but if we don't mention commit or rollback explicitly
        then rollback will be appended at the end.
        We can execute only retrieval query with above transaction block.If we try to insert or update data
        it will be rolled back.That's why it is necessary to use commit when we are executing
        Data Manipulation Langiage(DML) or Data Definition Language(DDL) Query.
        """
        try:
            with self.engine.connect() as conn:
                if values is not None:
                    result = conn.execute(text(stmnt),values)
                else:
                    result = conn.execute(text(stmnt))
            return result
        except Exception as err:
            print(f'Failed to execute dql commands -- {err}')

    def execute_ddl_and_dml_commands(self,stmnt,values=None):
        """
        Method to execute DDL and DML commands
        here we have followed another approach without using the "with" clause
        """
        connection = self.engine.connect()
        trans = connection.begin()
        try:
            if values is not None:

                result = connection.execute(text(stmnt),values)
            else:
                result = connection.execute(text(stmnt))
            trans.commit()
            connection.close()
            print('Command executed successfully.')
        except Exception as err:
            trans.rollback()
            print(f'Failed to execute ddl and dml commands -- {err}')

# Question 4

In [4]:

#Defining Db Credentials
USER_NAME = "postgres"
PASSWORD = "postgres"
PORT = 5432
DATABASE_NAME = "dvdrental"
HOST = "localhost"

#Note - Database should be created before executing below operation
#Initializing SqlAlchemy Postgresql Db Instance

db = PostgresqlDB(user_name=USER_NAME,
                    password=PASSWORD,
                    host=HOST,port=PORT,
                    db_name=DATABASE_NAME)
engine = db.engine
print("Checking database connection...")
with engine.connect() as conn:
    print("Connection Successful!")

Checking database connection...
Connection Successful!


In [6]:
#Doing CRUD Operation Via SqlAlchemy

#Creating Table
# creation of the table flights with all the attributes
create_table_stmnt = "CREATE TABLE flights (id serial PRIMARY KEY, origin VARCHAR(100) NOT NULL, \
                      destination VARCHAR(100) NOT NULL, duration INTEGER NOT NULL);"
db.execute_ddl_and_dml_commands(create_table_stmnt)

Command executed successfully.


In [7]:
#Insertion Of Values

#Single Insertion
# insertion of values into the table flights
Values = {'origin':'New York','destination':'London', 'duration':415}
single_insert_stmnt = "INSERT INTO flights (origin, destination, duration) \
                            VALUES (:origin,:destination,:duration);"
db.execute_ddl_and_dml_commands(single_insert_stmnt,Values)

Command executed successfully.


In [8]:
#Bulk Insertion
Values = [{'origin':'Shanghai','destination':'Paris', 'duration':760},
         {'origin':'Istanbul','destination':'Tokyo', 'duration':700}]

for value in Values:
    single_insert_stmnt = "INSERT INTO flights (origin, destination, duration) \
                            VALUES (:origin,:destination,:duration);"
    db.execute_ddl_and_dml_commands(single_insert_stmnt,value)

Command executed successfully.
Command executed successfully.


In [9]:
#Retrieval Query
select_query_stmnt = "select * from flights;"
result_1 = db.execute_dql_commands(select_query_stmnt)

#Traversing Result
for row in result_1:
    print(f'Origin:{row.origin} Destination:{row.destination} Duration:{row.duration}')

print('------------------------------------------------------------')
#Retrieval Query With Condition
select_query_stmnt_with_condition = "select * from flights where origin like :origin;"
value = {'origin':'%' + 'New York' + '%'}
result_2 = db.execute_dql_commands(select_query_stmnt_with_condition,value)

#Traversing Result
for row in result_2:
    print(f'Origin:{row.origin} Destination:{row.destination} Duration:{row.duration}')

Origin:New York Destination:London Duration:415
Origin:Shanghai Destination:Paris Duration:760
Origin:Istanbul Destination:Tokyo Duration:700
------------------------------------------------------------
Origin:New York Destination:London Duration:415


In [10]:
#Update Query
update_query_stmnt = "UPDATE flights SET duration=:new_duration WHERE duration=:duration"
value = [{'duration':700,'new_duration':500},{'duration':435,'new_duration':600}]
db.execute_ddl_and_dml_commands(update_query_stmnt,value)

Command executed successfully.


In [11]:
#Delete Query
delete_query_stmnt = "delete from flights WHERE duration=:duration"
value = {'duration':500}
db.execute_ddl_and_dml_commands(delete_query_stmnt,value)

Command executed successfully.


# Question 5: Find the title of films that have never been rented 

In [18]:
select_query_stmnt = "select f.title from film f left join inventory i on f.film_id = i.film_id left join rental r on i.inventory_id = r.inventory_id group by f.film_id having count(r.rental_id) = 0";

result_1 = db.execute_dql_commands(select_query_stmnt)

#Traversing Result
cnt = 0
for row in result_1:
    cnt +=1
    print(row)
print(cnt)

('Tadpole Park',)
('Sky Miracle',)
('Catch Amistad',)
('Treasure Command',)
('Muppet Mile',)
('Butch Panther',)
('Hocus Frida',)
('Firehouse Vietnam',)
('Kentuckian Giant',)
('Wake Jaws',)
('Kill Brotherhood',)
('Crowds Telemark',)
('Rainbow Shock',)
('Psycho Shrunk',)
('Suicides Silence',)
('Volume House',)
('Order Betrayed',)
('Argonauts Town',)
('Perdition Fargo',)
('Roof Champion',)
('Boondock Ballroom',)
('Ark Ridgemont',)
('Gladiator Westward',)
('Dazed Punk',)
('Raiders Antitrust',)
('Crossing Divorce',)
('Floats Garden',)
('Chinatown Gladiator',)
('Arsenic Independence',)
('Frankenstein Stranger',)
('Apollo Teen',)
('Gump Date',)
('Commandments Express',)
('Alice Fantasia',)
('Villain Desperate',)
('Crystal Breaking',)
('Hate Handicap',)
('Walls Artist',)
('Sister Freddy',)
('Pearl Destiny',)
('Deliverance Mulholland',)
('Chocolate Duck',)
42


# Question 6:  Find the first name, last name, and email of customers who live in the city 'London'.     

In [15]:
select_query_stmnt = "select first_name,last_name,email,ct.city from customer c join address a on c.address_id = a.address_id join city ct on a.city_id = ct.city_id where ct.city = 'London'";

result_1 = db.execute_dql_commands(select_query_stmnt)

#Traversing Result
for row in result_1:
    print(row)

('Mattie', 'Hoffman', 'mattie.hoffman@sakilacustomer.org', 'London')
('Cecil', 'Vines', 'cecil.vines@sakilacustomer.org', 'London')


# Question 7: Given a staff_id, find the total number of rentals processed by that staff member

In [16]:
select_query_stmnt = "select st.staff_id,count(r.rental_id) from staff st join rental r on st.staff_id = r.staff_id group by st.staff_id order by st.staff_id;";

result_1 = db.execute_dql_commands(select_query_stmnt)

#Traversing Result
for row in result_1:
    print(row)

(1, 8040)
(2, 8004)


# ORM and Transactions

In many practical applications, we generally use ORM to communicate with a DBMS. ORM stands for Object Relational Mapper, it allows us to interact with the database using high level objects defined by the Object-Oriented programming paradigm.

In [ ]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)
s = Session()

Session instance allows one to omit the text based query paradigm we followed above


In [ ]:
from sqlalchemy import Column, String, Integer, Boolean, ForeignKey, CheckConstraint
from sqlalchemy.orm import declarative_base, relationship

In [ ]:
Base = declarative_base()

In [ ]:
class Employee(Base): # inherit the Base ORM class
    __tablename__ = 'employee'
    emp_id = Column(Integer(), primary_key=True)
    name = Column(String(), nullable= False)
    dept_id = Column(Integer(), ForeignKey('department.dept_id'),nullable=False)

class Department(Base):
    __tablename__ = "department"
    dept_id = Column(Integer(), primary_key= True)
    name = Column(String(), nullable=False)
    works_in = relationship('Employee', backref='emp')

Base.metadata.create_all(engine)

In [ ]:
dept1 = Department( # create 2 department rows
            dept_id = 1,
            name = 'Konohagakure',
        )
dept2 = Department(
        dept_id = 2,
        name = 'Amegakure',
    )
emp1 = Employee(
        emp_id = 1000,
        name = 'Naruto',
        dept_id = dept1 # Naruto belongs to Department 1
    )
emp2 = Employee(
        emp_id = 2000,
        name = 'Nagato',
        dept_id = dept2 # Nagato belongs to Department 2
    )

In [ ]:
s.add(dept1)

In [ ]:
for _ in s.query(Department).all():
    print(_.name)
# dept1 has been added to the session, but has not been commited to the database yet
# It can still be viewed as part of a query from this session

In [ ]:
s.commit() # committing the changes

In [ ]:
s2 = Session()
s2.autoflush = False # by default each add statement adds the update to the session

In [ ]:
s2.add(dept2)

In [ ]:
for _ in s2.query(Department).all():
    print(_.name)
# The new department is not returned as part of the query
# as it has not been flushed yet i.e not added to the session
# Session 2 is same as Session 1 at this point

In [ ]:
s2.flush() # the changes has been flushed to the session

In [ ]:
for _ in s2.query(Department).all():
    print(_.name)

In [ ]:
# however the changes has not been made to the database,
# therefore we can rollback and can reverse the operation
# of adding dept2
s2.rollback()

In [ ]:
for _ in s2.query(Department).all():
    print(_.name)

### References
* https://docs.sqlalchemy.org/en/14/tutorial/index.html
* https://www.geeksforgeeks.org/sql-ddl-dql-dml-dcl-tcl-commands/
* Using SQLAlchemy as a ORM Set-Up
    * https://www.compose.com/articles/using-postgresql-through-sqlalchemy/
    * https://docs.sqlalchemy.org/en/13/orm/session_basics.html#when-do-i-make-a-sessionmaker
* Difference between SqlAlchemy Engine,Connection And Session
    * https://stackoverflow.com/questions/34322471/sqlalchemy-engine-connection-and-session-difference
* Accessing Postgresql via Java
    * https://www.javaguides.net/2020/02/java-crud-operations-with-postgresql.html
    * https://zetcode.com/java/postgresql/